# 01 · Ingest Bronze — Accounts

**Layer:** Bronze &nbsp;|&nbsp; **Stage:** ingestion &nbsp;|&nbsp; **Target:** `banking_bronze.raw_accounts`

Lands the account master feed exactly as the source system hands it over. The
bronze contract is *no cleansing* — records are written verbatim, warts and
all, with only lineage columns appended. Everything questionable in this feed
(duplicate account ids from the source's replay, missing branch codes, mixed
case status values) is preserved deliberately so the validation and silver
stages have something real to resolve.

Runs in parallel with `02_ingest_bronze_transactions`.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType,
)

spark = SparkSession.builder.appName("medallion-bronze-accounts").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
BRONZE_DB = "banking_bronze"
SOURCE_SYSTEM = "core-banking-cbs"
SOURCE_FILE   = f"s3a://bank-landing/accounts/dt={RUN_DATE}/accounts.csv"

In [ ]:
# In production this cell reads the landed CSV drop:
#
#     raw = (spark.read
#              .option("header", True)
#              .schema(ACCOUNT_SCHEMA)
#              .csv(SOURCE_FILE))
#
# For a self-contained demo we synthesise a feed with the same schema and the
# same defects the real one exhibits.

ACCOUNT_SCHEMA = StructType([
    StructField("account_id",     StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("account_type",   StringType(),  True),
    StructField("branch_code",    StringType(),  True),
    StructField("currency",       StringType(),  True),
    StructField("opening_balance", DoubleType(), True),
    StructField("status",         StringType(),  True),
    StructField("opened_on",      StringType(),  True),
    StructField("risk_band",      IntegerType(), True),
])

In [ ]:
rows = [
    ("ACC-1001", "CUST-501", "CURRENT", "BR-DEL-01", "INR",  125000.00, "ACTIVE",   "2019-04-11", 2),
    ("ACC-1002", "CUST-502", "SAVINGS", "BR-MUM-04", "INR",   48250.50, "ACTIVE",   "2020-01-23", 1),
    ("ACC-1003", "CUST-503", "CURRENT", "BR-BLR-02", "USD",   12000.00, "ACTIVE",   "2021-07-02", 3),
    ("ACC-1004", "CUST-504", "SAVINGS", "BR-DEL-01", "INR",     900.00, "DORMANT",  "2018-11-30", 4),
    ("ACC-1005", "CUST-505", "SALARY",  "BR-HYD-03", "INR",   76500.25, "ACTIVE",   "2022-02-14", 1),
    ("ACC-1006", "CUST-506", "CURRENT", "BR-MUM-04", "EUR",   33000.00, "ACTIVE",   "2020-09-19", 2),
    ("ACC-1007", "CUST-507", "SAVINGS", "BR-BLR-02", "INR",   15750.75, "active",   "2023-03-08", 1),
    ("ACC-1008", "CUST-508", "CURRENT", None,        "INR",   58000.00, "ACTIVE",   "2019-12-01", 3),
    ("ACC-1009", "CUST-509", "SALARY",  "BR-HYD-03", "INR",   91200.00, "CLOSED",   "2017-06-21", 5),
    ("ACC-1010", "CUST-510", "SAVINGS", "BR-DEL-01", "INR",    6400.00, "ACTIVE",   "2024-05-30", 2),
    # --- defects the source system is known to emit -----------------------
    ("ACC-1003", "CUST-503", "CURRENT", "BR-BLR-02", "USD",   12000.00, "ACTIVE",   "2021-07-02", 3),  # replayed duplicate
    ("ACC-1011", None,       "SAVINGS", "BR-MUM-04", "INR",    2200.00, "ACTIVE",   "2023-08-17", 1),  # orphan: no customer
    (None,       "CUST-512", "CURRENT", "BR-DEL-01", "INR",   44000.00, "ACTIVE",   "2022-10-05", 2),  # null primary key
]

raw = spark.createDataFrame(rows, schema=ACCOUNT_SCHEMA)
print(f"landed {raw.count()} account records from {SOURCE_SYSTEM}")

In [ ]:
# Lineage columns - the only thing bronze is allowed to add.
bronze_accounts = (
    raw
    .withColumn("_batch_id",      F.lit(BATCH_ID))
    .withColumn("_run_date",      F.lit(RUN_DATE))
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("_source_file",   F.lit(SOURCE_FILE))
    .withColumn("_ingested_at",   F.current_timestamp())
)

bronze_accounts.printSchema()

In [ ]:
(
    bronze_accounts.write
    .mode("overwrite")
    .format("parquet")
    .saveAsTable(f"{BRONZE_DB}.raw_accounts")
)

print(f"wrote {BRONZE_DB}.raw_accounts")

In [ ]:
spark.table(f"{BRONZE_DB}.raw_accounts").show(15, truncate=False)

print("row count :", spark.table(f"{BRONZE_DB}.raw_accounts").count())